# 🗄️ Course 6 — PostgreSQL Summary Stats and Window Functions

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Database:** Summer Olympics (1896–2012)

---

## 📋 About This Course

This course dives deep into window functions and advanced PostgreSQL features. Using the Summer Olympics dataset (medals, athletes, countries, events), it covers row numbering, fetching, ranking, paging, aggregate frames, pivoting with CROSSTAB, ROLLUP/CUBE subtotals, and string aggregation.

---

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | Introduction to Window Functions |
| Chapter 2 | Fetching, Ranking & Paging |
| Chapter 3 | Aggregate Window Functions & Frames |
| Chapter 4 | Beyond Window Functions: PIVOT, ROLLUP, CUBE, STRING_AGG |

---

## 📌 Chapter 1 — Introduction to Window Functions

---

### Numbering Rows
Assign a sequential number to each row in the dataset.

In [ ]:
SELECT
  *,
  ROW_NUMBER() OVER() AS Row_N
FROM Summer_Medals
ORDER BY Row_N ASC;

### Numbering Olympic Games — Ascending
Assign a number to each distinct year the Summer Olympics were held.

In [ ]:
SELECT
  Year,
  ROW_NUMBER() OVER() AS Row_N
FROM (
  SELECT DISTINCT year
  FROM Summer_Medals
  ORDER BY Year ASC) AS Years
ORDER BY Year ASC;

### Numbering Olympic Games — Descending
Assign row numbers so the most recent games have the lowest number.

In [ ]:
SELECT
  Year,
  ROW_NUMBER() OVER (ORDER BY YEAR DESC) AS Row_N
FROM (
  SELECT DISTINCT Year
  FROM Summer_Medals
) AS Years
ORDER BY Year;

### Ranking Athletes by Medals
Count medals per athlete, then rank them using ROW_NUMBER in a CTE.

In [ ]:
WITH Athlete_Medals AS (
  SELECT Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  GROUP BY Athlete)

SELECT
  athlete,
  ROW_NUMBER() OVER (ORDER BY Medals DESC) AS Row_N
FROM Athlete_Medals
ORDER BY Medals DESC;

### Reigning Champions — LAG()
Find the previous year's gold medalist in Men's 69KG Weightlifting using LAG().

In [ ]:
WITH Weightlifting_Gold AS (
  SELECT Year, Country AS champion
  FROM Summer_Medals
  WHERE Discipline = 'Weightlifting'
    AND Event = '69KG'
    AND Gender = 'Men'
    AND Medal = 'Gold')

SELECT
  Year, Champion,
  LAG(Champion, 1) OVER (ORDER BY YEAR ASC) AS Last_Champion
FROM Weightlifting_Gold
ORDER BY Year ASC;

### Reigning Champions by Gender — PARTITION BY
Use PARTITION BY gender to keep each gender's champion history separate.

In [ ]:
WITH Tennis_Gold AS (
  SELECT DISTINCT Gender, Year, Country
  FROM Summer_Medals
  WHERE Year >= 2000
    AND Event = 'Javelin Throw'
    AND Medal = 'Gold')

SELECT
  Gender, Year,
  Country AS Champion,
  LAG(Country, 1) OVER (PARTITION BY gender
            ORDER BY year ASC) AS Last_Champion
FROM Tennis_Gold
ORDER BY Gender ASC, Year ASC;

### Reigning Champions by Gender and Event
Partition by both gender and event to avoid mixing results across categories.

In [ ]:
WITH Athletics_Gold AS (
  SELECT DISTINCT Gender, Year, Event, Country
  FROM Summer_Medals
  WHERE Year >= 2000
    AND Discipline = 'Athletics'
    AND Event IN ('100M', '10000M')
    AND Medal = 'Gold')

SELECT
  Gender, Year, Event,
  Country AS Champion,
  LAG(Country, 1) OVER (PARTITION BY gender, event
            ORDER BY Year ASC) AS Last_Champion
FROM Athletics_Gold
ORDER BY Event ASC, Gender ASC, Year ASC;

### ❓ Quiz: ROW_NUMBER with PARTITION BY
> **Q:** Running `ROW_NUMBER() OVER (PARTITION BY Year ORDER BY Medals DESC)` — what row number would Iran (2008, 27 medals) get?

| Year | Country | Medals |
|------|---------|--------|
| 2004 | IRN | 32 | 2004 | LBN | 17 | 2004 | KSA | 4 |
| 2008 | IRQ | 29 | **2008** | **IRN** | **27** | 2008 | UAE | 12 |

> **A: 2** — resets at each year, ranks within 2008: IRQ=1, IRN=2, UAE=3.

---

## 📌 Chapter 2 — Fetching, Ranking & Paging

---

### LEAD() — Future Gold Medalists
For each year, fetch the current and the gold medalist 3 competitions ahead.

In [ ]:
WITH Discus_Medalists AS (
  SELECT DISTINCT Year, Athlete
  FROM Summer_Medals
  WHERE Medal = 'Gold'
    AND Event = 'Discus Throw'
    AND Gender = 'Women'
    AND Year >= 2000)

SELECT
  year, Athlete,
  LEAD(Athlete, 3) OVER (ORDER BY year ASC) AS Future_Champion
FROM Discus_Medalists
ORDER BY Year ASC;

### FIRST_VALUE() — First Athlete Alphabetically
Return all athletes alongside the first athlete ordered alphabetically.

In [ ]:
WITH All_Male_Medalists AS (
  SELECT DISTINCT Athlete
  FROM Summer_Medals
  WHERE Medal = 'Gold' AND Gender = 'Men')

SELECT
  athlete,
  FIRST_VALUE(athlete) OVER (
    ORDER BY athlete ASC
  ) AS First_Athlete
FROM All_Male_Medalists;

### LAST_VALUE() — Last Olympic Host City
Return each year's host city and the last city ever to host the Olympics.  
> **Note:** `RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` is needed so LAST_VALUE looks across the entire window, not just up to the current row.

In [ ]:
WITH Hosts AS (
  SELECT DISTINCT Year, City FROM Summer_Medals)

SELECT
  Year, City,
  LAST_VALUE(City) OVER (
   ORDER BY year ASC
   RANGE BETWEEN
     UNBOUNDED PRECEDING AND
     UNBOUNDED FOLLOWING
  ) AS Last_City
FROM Hosts
ORDER BY Year ASC;

### RANK() — Athletes by Medals
Rank athletes by medals earned, assigning the same rank to ties (with gaps).

In [ ]:
WITH Athlete_Medals AS (
  SELECT Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  GROUP BY Athlete)

SELECT
  Athlete, Medals,
  RANK() OVER (ORDER BY Medals DESC) AS Rank_N
FROM Athlete_Medals
ORDER BY Medals DESC;

### DENSE_RANK() — Athletes by Country
Rank athletes within each country without skipping numbers on ties.

In [ ]:
WITH Athlete_Medals AS (
  SELECT Country, Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country IN ('JPN', 'KOR') AND Year >= 2000
  GROUP BY Country, Athlete
  HAVING COUNT(*) > 1)

SELECT
  Country, Athlete,
  DENSE_RANK() OVER (PARTITION BY Country
                ORDER BY Medals DESC) AS Rank_N
FROM Athlete_Medals
ORDER BY Country ASC, RANK_N ASC;

### ❓ Quiz: DENSE_RANK
> **Q:** Given this table ranked by Medals DESC, what rank would BHR (7 medals) get with DENSE_RANK?

| Country | Medals | DENSE_RANK |
|---------|--------|------------|
| IRN | 23 | 1 |
| IRQ | 19 | 2 |
| LBN | 19 | 2 |
| SYR | 19 | 2 |
| **BHR** | **7** | **3** |
| KSA | 3 | 4 |

> **A: 3** — DENSE_RANK doesn't skip numbers, so after three countries tied at rank 2, BHR gets rank 3.

### NTILE() — Paging Events
Split 666 distinct Olympic events into 111 equal groups.

In [ ]:
WITH Events AS (
  SELECT DISTINCT Event FROM Summer_Medals)

SELECT
  event,
  NTILE(111) OVER (ORDER BY Event ASC) AS Page
FROM Events
ORDER BY Event ASC;

### NTILE() — Top/Middle/Bottom Thirds
Split athletes into thirds by medal count, then calculate the average medals per third.

In [ ]:
WITH Athlete_Medals AS (
  SELECT Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  GROUP BY Athlete
  HAVING COUNT(*) > 1),

  Thirds AS (
  SELECT Athlete, Medals,
    NTILE(3) OVER (ORDER BY Medals DESC) AS Third
  FROM Athlete_Medals)

SELECT
  Third,
  AVG(Medals) AS Avg_Medals
FROM Thirds
GROUP BY Third
ORDER BY Third;

---

## 📌 Chapter 3 — Aggregate Window Functions & Frames

---

### Running Total — SUM() OVER
Calculate the running total of gold medals earned by USA athletes since 2000.

In [ ]:
WITH Athlete_Medals AS (
  SELECT Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country = 'USA' AND Medal = 'Gold' AND Year >= 2000
  GROUP BY Athlete)

SELECT
  athlete, medals,
  SUM(medals) OVER (ORDER BY athlete ASC) AS Max_Medals
FROM Athlete_Medals
ORDER BY Athlete ASC;

### Running Maximum — MAX() OVER with PARTITION BY
Track each country's maximum gold medals earned so far across years.

In [ ]:
WITH Country_Medals AS (
  SELECT Year, Country, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country IN ('CHN', 'KOR', 'JPN')
    AND Medal = 'Gold' AND Year >= 2000
  GROUP BY Year, Country)

SELECT
  Year, country, medals,
  MAX(Medals) OVER (PARTITION BY country
                ORDER BY year ASC) AS Max_Medals
FROM Country_Medals
ORDER BY Country ASC, Year ASC;

### Running Minimum — MIN() OVER
Track France's minimum gold medals earned so far, year by year.

In [ ]:
WITH France_Medals AS (
  SELECT Year, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country = 'FRA' AND Medal = 'Gold' AND Year >= 2000
  GROUP BY Year)

SELECT
  year, Medals,
  MIN(medals) OVER (ORDER BY year ASC) AS Min_Medals
FROM France_Medals
ORDER BY Year ASC;

### Moving Maximum — ROWS BETWEEN
Compare only current and next year's medals for Scandinavian countries.

In [ ]:
WITH Scandinavian_Medals AS (
  SELECT Year, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country IN ('DEN', 'NOR', 'FIN', 'SWE', 'ISL')
    AND Medal = 'Gold'
  GROUP BY Year)

SELECT
  year, medals,
  MAX(Medals) OVER (ORDER BY year ASC
             ROWS BETWEEN CURRENT ROW
             AND 1 FOLLOWING) AS Max_Medals
FROM Scandinavian_Medals
ORDER BY Year ASC;

### Moving Maximum — Last 2 and Current Row
Compare the last two and current athletes' medals for Chinese gold medalists.

In [ ]:
WITH Chinese_Medals AS (
  SELECT Athlete, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country = 'CHN' AND Medal = 'Gold' AND Year >= 2000
  GROUP BY Athlete)

SELECT
  Athlete, Medals,
  MAX(Medals) OVER (ORDER BY Athlete ASC
            ROWS BETWEEN 2 PRECEDING
            AND CURRENT ROW) AS Max_Medals
FROM Chinese_Medals
ORDER BY Athlete ASC;

### 3-Year Moving Average
Calculate Russia's 3-year moving average of gold medals since 1980.

In [ ]:
WITH Russian_Medals AS (
  SELECT Year, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Country = 'RUS' AND Medal = 'Gold' AND Year >= 1980
  GROUP BY Year)

SELECT
  Year, Medals,
  AVG(Medals) OVER
    (ORDER BY Year ASC
     ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS Medals_MA
FROM Russian_Medals
ORDER BY Year ASC;

### 3-Game Moving Total by Country
Calculate each country's 3-game moving total, partitioned to avoid mixing countries.

In [ ]:
WITH Country_Medals AS (
  SELECT Year, Country, COUNT(*) AS Medals
  FROM Summer_Medals
  GROUP BY Year, Country)

SELECT
  Year, Country, Medals,
  SUM(Medals) OVER
    (PARTITION BY Country
     ORDER BY Year ASC
     ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS Medals_MA
FROM Country_Medals
ORDER BY Country ASC, Year ASC;

---

## 📌 Chapter 4 — Beyond Window Functions

---

### CROSSTAB — Basic Pivot
Pivot Pole Vault gold medalists by year to get gender as rows and years as columns.

In [ ]:
CREATE EXTENSION IF NOT EXISTS tablefunc;

SELECT * FROM CROSSTAB($$
  SELECT Gender, Year, Country
  FROM Summer_Medals
  WHERE Year IN (2008, 2012)
    AND Medal = 'Gold'
    AND Event = 'Pole Vault'
  ORDER By Gender ASC, Year ASC;
$$) AS ct (Gender VARCHAR,
           "2008" VARCHAR,
           "2012" VARCHAR)
ORDER BY Gender ASC;

### CROSSTAB with RANK() — Pivoting Rankings
Rank FRA, GBR, GER by gold medals per year, then pivot results by year.

In [ ]:
-- Step 1: Count gold medals per country and year
SELECT country, year, COUNT(*) AS Awards
FROM Summer_Medals
WHERE Country IN ('FRA', 'GBR', 'GER')
  AND Year IN (2004, 2008, 2012)
  AND Medal = 'Gold'
GROUP BY country, year
ORDER BY Country ASC, Year ASC;

In [ ]:
-- Step 2: Rank countries per year
WITH Country_Awards AS (
  SELECT Country, Year, COUNT(*) AS Awards
  FROM Summer_Medals
  WHERE Country IN ('FRA', 'GBR', 'GER')
    AND Year IN (2004, 2008, 2012)
    AND Medal = 'Gold'
  GROUP BY Country, Year)

SELECT country, year,
  RANK() OVER(PARTITION BY Year ORDER BY Awards DESC)::INTEGER AS rank
FROM Country_Awards
ORDER BY Country ASC, Year ASC;

In [ ]:
-- Step 3: Pivot by year using CROSSTAB
CREATE EXTENSION IF NOT EXISTS tablefunc;

SELECT * FROM CROSSTAB($$
  WITH Country_Awards AS (
    SELECT Country, Year, COUNT(*) AS Awards
    FROM Summer_Medals
    WHERE Country IN ('FRA', 'GBR', 'GER')
      AND Year IN (2004, 2008, 2012)
      AND Medal = 'Gold'
    GROUP BY Country, Year)
  SELECT Country, Year,
    RANK() OVER (PARTITION BY Year ORDER BY Awards DESC)::INTEGER AS rank
  FROM Country_Awards
  ORDER BY Country ASC, Year ASC;
$$) AS ct (Country VARCHAR,
           "2004" INTEGER,
           "2008" INTEGER,
           "2012" INTEGER)
ORDER BY Country ASC;

### ROLLUP — Country-Level Subtotals
Count gold medals per country and gender for Scandinavia in 2004, with country-level subtotals.

In [ ]:
SELECT country, gender, COUNT(*) AS Gold_Awards
FROM Summer_Medals
WHERE Year = 2004
  AND Medal = 'Gold'
  AND Country IN ('DEN', 'NOR', 'SWE')
GROUP BY country, ROLLUP(gender)
ORDER BY Country ASC, Gender ASC;

### CUBE — All Group-Level Subtotals
Break down Russia's 2012 medals by gender and type, including all subtotals and grand total.

In [ ]:
SELECT Gender, Medal, COUNT(*) AS Awards
FROM Summer_Medals
WHERE Year = 2012 AND Country = 'RUS'
GROUP BY CUBE(Gender, Medal)
ORDER BY Gender ASC, Medal ASC;

### COALESCE — Cleaning Up NULLs
Replace NULLs from ROLLUP with meaningful labels using COALESCE.

In [ ]:
SELECT
  COALESCE(Country, 'All countries') AS Country,
  COALESCE(Gender, 'All genders') AS Gender,
  COUNT(*) AS Awards
FROM Summer_Medals
WHERE Year = 2004 AND Medal = 'Gold'
  AND Country IN ('DEN', 'NOR', 'SWE')
GROUP BY ROLLUP(Country, Gender)
ORDER BY Country ASC, Gender ASC;

### STRING_AGG — Summarizing Results
Rank countries by 2000 Olympic gold medals, then compress the top 3 into a single comma-separated string.

In [ ]:
WITH Country_Medals AS (
  SELECT Country, COUNT(*) AS Medals
  FROM Summer_Medals
  WHERE Year = 2000 AND Medal = 'Gold'
  GROUP BY Country),

  Country_Ranks AS (
  SELECT Country,
    RANK() OVER (ORDER BY Medals DESC) AS Rank
  FROM Country_Medals
  ORDER BY Rank ASC)

SELECT STRING_AGG(Country, ', ')
FROM Country_Ranks
WHERE Rank <= 3;

> **Output:** `USA, RUS, AUS`